> ## SOLUTION / ANSWER KEY &mdash; Lab 6.4
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-04-assemble-agent.ipynb`](../lab-04-assemble-agent.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.4 &mdash; Assemble a ReAct Agent (the four pieces)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Complete create_react_agent so it registers tools by name
- Assemble a model + a tools list + a prompt into one agent
- Inspect the bound agent: which tools and model it holds

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps use a tiny **LangChain-shaped shim** (same names &amp; shapes as real LangChain) so they run offline &amp; deterministically. Advanced labs end with an **optional** cell that runs the **real** library.

**Reference:** [Module 6 slides &mdash; Assemble a ReAct agent](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-06-04"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
A LangChain agent is four pieces bound together (deck slide 9): a **model**, a **tools** list, a
**prompt**, and **`create_react_agent(model, tools, prompt)`** that ties them into a reasoning core.
`create_react_agent` registers the tools **by name** so the loop can look each one up.

In [ ]:
# --- LangChain-SHAPED shim: a tool has .name, .description (from the docstring), .args, .invoke() ---
import inspect
class Tool:
    def __init__(self, fn, name=None, description=None):
        self.fn = fn
        self.name = name or fn.__name__
        self.description = (description or inspect.getdoc(fn) or "").strip()
        self.args = list(inspect.signature(fn).parameters)
    def invoke(self, value):
        return self.fn(**value) if isinstance(value, dict) else self.fn(value)
    def __repr__(self):
        return "Tool(name=%r)" % self.name
def tool(fn):
    """The @tool decorator -- wrap a plain function into a Tool (mirrors langchain_core.tools.tool)."""
    return Tool(fn)

class AIMessage:
    def __init__(self, content): self.content = content
class FakeChatModel:
    """Deterministic stand-in for ChatOllama / ChatGroq: replays a scripted list of replies.
    Real code: from langchain_ollama import ChatOllama; model = ChatOllama(model="llama3.2:1b").
    Like the real thing, .invoke(prompt) returns a message whose text is in .content."""
    def __init__(self, script): self.script = list(script); self.i = 0
    def invoke(self, prompt):
        reply = self.script[min(self.i, len(self.script) - 1)]; self.i += 1
        return AIMessage(reply)

class PromptTemplate:
    """Mirrors LangChain: PromptTemplate.from_template(...).format(input=..., ...)."""
    def __init__(self, template): self.template = template
    @classmethod
    def from_template(cls, template): return cls(template)
    def format(self, **kw):
        s = self.template
        for k, v in kw.items():
            s = s.replace("{" + k + "}", str(v))
        return s

@tool
def calculator(expression):
    """Do exact arithmetic."""
    return "(computed)"
@tool
def lookup(key):
    """Look up a known fact by key."""
    return "(fact)"
print("two tools ready:", calculator.name, "&", lookup.name)

## Your Turn
Finish `create_react_agent` (register tools by name), then assemble the agent in `build_agent`.

In [ ]:
def create_react_agent(model, tools, prompt):
    """Bind model + tools + prompt into an agent (mirrors langchain.agents.create_react_agent)."""
    return {"model": model, "tools": {t.name: t for t in tools}, "prompt": prompt}

def build_agent():
    model  = FakeChatModel(["Final Answer: ok"])
    prompt = PromptTemplate.from_template("{scratchpad}\nQuestion: {input}")
    tools  = [calculator, lookup]
    return create_react_agent(model, tools, prompt)

try:
    agent = build_agent()
    print('registered tools:', list(agent['tools']))
    print('model bound      :', type(agent['model']).__name__)
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the agent binds model, tools and prompt", lambda: {"model", "tools", "prompt"} <= set(build_agent()))
expect_true("both tools are registered by name", lambda: set(build_agent()["tools"]) == {"calculator", "lookup"})
expect_true("a registered entry is a Tool", lambda: build_agent()["tools"]["calculator"].name == "calculator")
expect_true("the model is bound", lambda: build_agent()["model"] is not None)
expect_true("create_react_agent maps a single tool too", lambda: list(create_react_agent(FakeChatModel(["x"]), [calculator], None)["tools"]) == ["calculator"])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
model + tools + prompt -> create_react_agent = a bound agent. It knows its tools by name; next, the executor runs the loop over them.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>